# UK Biobank Data

---

### package imports and basic functions

---

In [45]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
%load_ext autoreload
%autoreload 2

# Path to add to src folder (to use local version of spectranorm)
sys.path.append(os.path.abspath("/mountpoint/code/projects/spectranorm/package/spectranorm/src/"))

from spectranorm import snm


In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Data access via Mediaflux

---

Note: Following UKB exemption request approval, freesurfer data was downloaded and used in this notebook.


## Extracting data

---

In [4]:
ukb_dir = '/home/ubuntu/mediafluxmount/Datasets/UK_Biobank/Subjects/'
ukb_subjects = [x.split('/')[-1] for x in list_dirs(ukb_dir)]
len(ukb_subjects)


63005

In [13]:
# make a list of all instances with freesurfer output provided
ukb_unique_ids = []
for sub in tqdm(ukb_subjects):
    test_path = f"{ukb_dir}{sub}"
    ukb_unique_ids.extend(['-'.join([sub, x.split('/')[-1]]) for x in list_dirs(test_path) if '20263_' in x])
len(ukb_unique_ids)
    

  0%|          | 0/63005 [00:00<?, ?it/s]

64495

In [20]:
def zip_contains_all(zip_path, items_to_check):
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = set(zf.namelist())
        return all(item in names for item in items_to_check)


In [27]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]

ukb_unique_ids_with_fs = []

for uuid in tqdm(ukb_unique_ids):
    sub, ses = uuid.split('-')
    zip_path = f"{ukb_dir}{sub}/{ses}/{sub}_{ses}.zip"
    
    if not os.path.exists(zip_path):
        continue  # skip if file doesn't exist
    
    try:
        if zip_contains_all(zip_path, [f"FreeSurfer/surf/{x}" for x in items]):
            ukb_unique_ids_with_fs.append(uuid)
    except zipfile.BadZipFile:
        # skip corrupted zip files
        continue

len(ukb_unique_ids_with_fs)

  0%|          | 0/64495 [00:00<?, ?it/s]

64491

In [28]:
np.save(
    ensure_dir("/mountpoint/data/normative/datasets/UKB/subjects.npy"),
    np.array(ukb_unique_ids_with_fs)
)


In [29]:
ukb_subjects_with_fs = list(set([x.split('-')[0] for x in ukb_unique_ids_with_fs]))
len(ukb_subjects_with_fs)


59865

In [30]:
items = [
    "lh.white", "rh.white",
    "lh.pial", "rh.pial",
    "lh.thickness", "rh.thickness",
    "lh.orig.nofix", "rh.orig.nofix",
    "lh.sphere.reg", "rh.sphere.reg",
]
ukb_valid_subjects = ukb_unique_ids_with_fs
len(ukb_valid_subjects)


64491

In [31]:
# ignore warning
nib.imageglobals.logger.setLevel(40)


In [ ]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(ukb_valid_subjects)):
    sub, ses = subject.split("-")
    sub_dir = f"{idx:02d}"[-2:]
    
    zip_path = f"{ukb_dir}{sub}/{ses}/{sub}_{ses}.zip"
    freesurfer_directory = f"/mountpoint/data/UKB/snm_thickness/freesurfer/{subject}/"
    
    thickness_fslr_output = f"/mountpoint/data/normative/fs_LR_32k/UKB/{sub_dir}/{subject}.thickness.fslr.npy"
    
    # Skip if thickness file already exists
    if os.path.exists(thickness_fslr_output):
        continue
    
    # 1. Extract required items into subject-specific directory
    try:
        with zipfile.ZipFile(zip_path, "r") as zf:
            for item in items:
                file_in_zip = f"FreeSurfer/surf/{item}"
                zf.extract(file_in_zip, path=freesurfer_directory)
    except zipfile.BadZipFile as e:
        print(f"Skipping {subject}, corrupt ZIP: {e}")
        continue

    # # 2. Compute fslr thickness
    transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(
        os.path.join(freesurfer_directory, "FreeSurfer")
    )
    np.save(
        ensure_dir(thickness_fslr_output),
        transformed_fslr_thickness.astype(np.float32)
    )
    
    # 3. Clean up directory
    shutil.rmtree(freesurfer_directory, ignore_errors=True)


UKB demography information

In [ ]:
# empty dict to hold information
ukb_valid_subjects_dict = {
    subject: {
        "participant_id": subject.split("-")[0],
        "session_id": subject.split("-")[1][-3:],
        "subject_index": idx,
    }
    for idx, subject in enumerate(tqdm(ukb_valid_subjects))
    if Path(f"/mountpoint/data/normative/fs_LR_32k/UKB/{(idx%100):02d}/{subject}.thickness.fslr.npy").exists()
}

len(ukb_valid_subjects_dict), list(ukb_valid_subjects_dict.items())[:3]


In [ ]:
participant_session_list_dict = {}

for key in ukb_valid_subjects_dict:
    session_list = participant_session_list_dict.get(ukb_valid_subjects_dict[key]["participant_id"], [])
    session_list.append(key)
    participant_session_list_dict[ukb_valid_subjects_dict[key]["participant_id"]] = session_list

len(participant_session_list_dict), list(participant_session_list_dict.items())[:3]


In [46]:
%%time

# ukbdf = pd.read_csv("/mountpoint/data/UKB/ukbexemptionscripts/ukb677892.csv")
# ukbdf.shape, ukbdf.head()
# It takes too long to load the complete df of ~60GB csv file, let's use a fast alternative

ukbdf = pl.scan_csv("/mountpoint/data/UKB/ukbexemptionscripts/ukb677892.csv")

# Number of columns is known
print("Columns:", len(ukbdf.columns))


<timed exec>:8: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.


Columns: 31692
CPU times: user 514 ms, sys: 32.8 ms, total: 547 ms
Wall time: 508 ms


In [ ]:
sex_coding = {0: "F", 1: "M"}

selected_columns = []

selected_columns.append("eid") # Subject ID

selected_columns.extend([x for x in ukbdf.columns if "20544" in x]) # Mental health problems ever diagnosed by a professional

selected_columns.append("31-0.0") # Sex coded as int

selected_columns.append("34-0.0") # year of birth
selected_columns.append("52-0.0") # month of birth

selected_columns.extend(["53-2.0", "53-3.0"]) # Date of attending assessment centre

selected_columns.extend(["54-2.0", "54-3.0"]) # UK Biobank assessment centre

selected_columns


In [79]:
%%time

# load a subset of the large df into memory
selected_ukbdf = pl.read_csv(
    "/mountpoint/data/UKB/ukbexemptionscripts/ukb677892.csv",
    columns=selected_columns,
)


CPU times: user 1min 2s, sys: 4.5 s, total: 1min 7s
Wall time: 19.4 s


In [80]:
selected_ukbdf.shape

(502188, 24)

In [95]:
# Sex as string
sex_coding = {0: "F", 1: "M"}

selected_ukbdf = selected_ukbdf.with_columns(
    pl.Series("sex", [sex_coding.get(x, None) for x in selected_ukbdf["31-0.0"]])
)


In [108]:
# convert birth year/month to fractional year
birth_frac = pl.col("34-0.0") + pl.col("52-0.0") / 12

# parse scan dates
scan_2 = pl.col("53-2.0").str.strptime(pl.Date, "%Y-%m-%d", strict=False)
scan_3 = pl.col("53-3.0").str.strptime(pl.Date, "%Y-%m-%d", strict=False)

# compute age at scan (fractional years)
selected_ukbdf = selected_ukbdf.with_columns([
    ((scan_2.dt.year() + scan_2.dt.month() / 12) - birth_frac).alias("age_2_0"),
    ((scan_3.dt.year() + scan_3.dt.month() / 12) - birth_frac).alias("age_3_0"),
])


In [ ]:
# summarize mental health diagnosis:
mh_diag_cols = [x for x in ukbdf.columns if "20544" in x]

# compute MH_diagnosis: True if ANY column is non-empty string
selected_ukbdf = selected_ukbdf.with_columns(
    pl.any_horizontal([pl.col(c) != "" for c in mh_diag_cols]).alias("MH_diagnosis")
)


In [122]:
selected_ukbdf = selected_ukbdf.rename({
    "54-2.0": "site_2_0",
    "54-3.0": "site_3_0",
})

In [ ]:
# convert to dictionary keyed by eid
eid_dict = {
    row["eid"]: row
    for row in selected_ukbdf[['eid', 'sex', 'age_2_0', 'age_3_0', "MH_diagnosis", "site_2_0", "site_3_0"]].to_dicts()}


In [ ]:
for key in ukb_valid_subjects_dict:
    ukb_valid_subjects_dict[key]["unique_id"] = key

for eid in eid_dict:
    for key in participant_session_list_dict.get(str(eid), []):
        ukb_valid_subjects_dict[key]["sex"] = eid_dict[eid]['sex']
        ukb_valid_subjects_dict[key]["age"] = eid_dict[eid][f'age_{key[-3:]}']
        ukb_valid_subjects_dict[key]["site"] = eid_dict[eid][f'site_{key[-3:]}']
        ukb_valid_subjects_dict[key]["validity_check"] = (eid_dict[eid]['MH_diagnosis'] == False)


len(ukb_valid_subjects_dict), list(ukb_valid_subjects_dict.items())[:1]


In [ ]:
%%time
for key in tqdm(ukb_valid_subjects_dict):
    idx = ukb_valid_subjects_dict[key]["subject_index"]
    subject = ukb_valid_subjects_dict[key]["unique_id"]
    ukb_valid_subjects_dict[key]["thickness"] = np.load(
        f"/mountpoint/data/normative/fs_LR_32k/UKB/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()

len(ukb_valid_subjects_dict), list(ukb_valid_subjects_dict.items())[:1]


In [ ]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(ukb_valid_subjects)):
    if (subject in ukb_valid_subjects_dict) and ("euler_no" not in ukb_valid_subjects_dict[subject]):
        sub, ses = subject.split("-")
        sub_dir = f"{idx:02d}"[-2:]

        zip_path = f"{ukb_dir}{sub}/{ses}/{sub}_{ses}.zip"
        freesurfer_directory = f"/mountpoint/data/UKB/snm_thickness/freesurfer/{subject}/"

        # 1. Extract required items into subject-specific directory
        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                for item in eno_items:
                    file_in_zip = f"FreeSurfer/surf/{item}"
                    zf.extract(file_in_zip, path=freesurfer_directory)
        except zipfile.BadZipFile as e:
            print(f"Skipping {subject}, corrupt ZIP: {e}")
            continue

        # 2. Compute euler number
        ukb_valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
            os.path.join(freesurfer_directory, "FreeSurfer")
        )

        # 3. Clean up directory
        shutil.rmtree(freesurfer_directory, ignore_errors=True)


In [151]:
import joblib

joblib.dump(ukb_valid_subjects_dict, ensure_dir("/mountpoint/data/normative/datasets/UKB/subjects.joblib"))


['/mountpoint/data/normative/datasets/UKB/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
ukb_valid_subjects_dict = joblib.load(
    "/mountpoint/data/normative/datasets/UKB/subjects.joblib"
)

len(ukb_valid_subjects_dict), list(ukb_valid_subjects_dict.items())[:1]


In [ ]:
ukb_df = pd.DataFrame({
    'age': [ukb_valid_subjects_dict[key]["age"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'thickness': [ukb_valid_subjects_dict[key]["thickness"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'sex': [ukb_valid_subjects_dict[key]["sex"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'site': [ukb_valid_subjects_dict[key]["site"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'subject_ID': [ukb_valid_subjects_dict[key]["participant_id"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'euler_no': [ukb_valid_subjects_dict[key]["euler_no"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'subject_folder': [ukb_valid_subjects_dict[key]["unique_id"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
    'subject_index': [ukb_valid_subjects_dict[key]["subject_index"] for key in ukb_valid_subjects_dict if ukb_valid_subjects_dict[key]["validity_check"]],
})
dataset_name = 'UKB'
ukb_df['dataset'] = dataset_name
ukb_df.head(), ukb_df.shape


In [159]:
# randomly select only one timepoint per subject (cross-sectional sample)
ukb_df_subset = ukb_df.groupby("subject_ID", group_keys=False).sample(n=1, random_state=1234)

# Keep only sites with at least 15 subjects
subjects_per_site = ukb_df_subset.groupby("site")["subject_ID"].nunique()
valid_sites = subjects_per_site[subjects_per_site >= 15].index

ukb_df_subset[ukb_df_subset["site"].isin(valid_sites)].to_parquet(
    ensure_dir(f'/mountpoint/data/normative/datasets/{dataset_name}/demography.parquet')
)

ukb_df_subset[ukb_df_subset["site"].isin(valid_sites)].shape


(48422, 9)

In [155]:
valid_sites

Index(['11025', '11026', '11027', '11028'], dtype='object', name='site')